# 05 — Controlled evaluation, interpretability, and field deployment/QC

| Item | Definition |
|---|---|
| **Scientific purpose** | Evaluate matched synthetic experiments, expose the graph mechanism, reconstruct whole images, and deploy the trained model to the field line with conservative QC. |
| **Inputs** | Notebook-03 test realizations and normalization; Notebook-04 controlled checkpoints; Stage-01 real AVO, RGT, low-frequency elastic model, coordinates, and local wells. |
| **Outputs** | Per-realization/summary/paired metrics, documented representative figures, whole-image predictions, field consistency plots, and model/prior sensitivity products. |
| **Data availability** | Evaluation code and figure definitions are public. Checkpoints and field/private-derived arrays remain local. |
| **Local/private-data requirements** | Completed controlled checkpoints plus licensed Stage-01 artifacts for field deployment. No fabricated metric placeholders are populated. |
| **Software requirements** | `pip install -e ".[field,ml,notebooks]"`. |
| **Approximate runtime** | Synthetic inference: minutes per variant/realization; field tiling and sensitivity scale with checkpoints and integration steps. |
| **Pipeline position** | Final stage consuming Notebooks 01, 03, and 04. |

Field wells contributed to upstream model construction. Results are therefore described as **field deployment and QC** or **field consistency assessment**, never independent field validation. Ensemble/checkpoint/prior-cutoff spread is sensitivity, not calibrated posterior uncertainty.

In [ ]:
from pathlib import Path

def find_repository_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sage_avo").exists():
            return candidate
    raise RuntimeError("Run this notebook from the installed SAGE-AVO repository.")

ROOT = find_repository_root()

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sage_avo.config import load_config, seed_everything
from sage_avo.evaluation import field_well_consistency
from sage_avo.evaluation.controlled import evaluate_controlled_ablation
from sage_avo.evaluation.inference import infer_full_realization, load_normalization
from sage_avo.evaluation.sensitivity import ensemble_sensitivity
from sage_avo.experiments.prediction import load_controlled_model, predict_controlled_variant
from sage_avo.forward import ForwardConfig, forward_avo_three_band
from sage_avo.forward.qc import compare_forward_outputs
from sage_avo.models import LEARNED_VARIANTS
from sage_avo.models.graph import build_rgt_edges
from sage_avo.models.sage_avo import angular_features
from sage_avo.structure.graph import GraphEdges
from sage_avo.visualization import plot_graph_mechanism, plot_inversion_comparison

workflow_path = ROOT / "configs" / "sage_avo_s01.yaml"
workflow = load_config(workflow_path)
paths_file = ROOT / "configs" / "paths.yaml"
if not paths_file.exists():
    raise FileNotFoundError("Create ignored configs/paths.yaml from configs/paths.example.yaml.")
paths = load_config(paths_file)
seed_everything(int(workflow["experiment"]["seed"]))

private_root = Path(paths["private_artifact_root"])
dataset_dir = private_root / "stage_artifacts" / "stage03" / "dataset"
experiment_dir = private_root / "stage_artifacts" / "stage04" / "experiments"
figure_dir = private_root / "figures" / "stage05"
figure_dir.mkdir(parents=True, exist_ok=True)

## Part A — Controlled synthetic evaluation

The required conditions are low-frequency-prior-only, full SAGE-AVO, no-GNN, no-RGT-steering, and no-physics-loss. Learned variants are trained on the same realization split and evaluated on complete test realizations. RMSE, MAE, R², SSIM, Dice/F1, and mIoU are first computed per realization; pooled summaries and paired realization-level bootstrap intervals are secondary.

HCTNet enters this table only after a matched retraining with identical prior, split, normalization, masks, tiling, and checkpoint rule. Otherwise it remains explicitly historical/non-controlled.

In [ ]:
if not (dataset_dir / "dataset_manifest.json").exists():
    raise FileNotFoundError("Run Notebook 03 first.")
checkpoints = {
    variant: experiment_dir / "runs" / variant / "best_sampling.pt"
    for variant in LEARNED_VARIANTS
}
checkpoint_status = pd.DataFrame([
    {"variant": variant, "checkpoint": path.name, "available": path.exists()}
    for variant, path in checkpoints.items()
])
display(checkpoint_status)
all_checkpoints_available = bool(checkpoint_status["available"].all())

### A1. Whole-test prediction generation

In [ ]:
run_predictions = os.getenv("SAGE_AVO_RUN_EVALUATION", "0") == "1"
if run_predictions:
    if not all_checkpoints_available:
        missing = [str(path) for path in checkpoints.values() if not path.exists()]
        raise FileNotFoundError("Controlled evaluation requested but checkpoints are missing:\n" + "\n".join(missing))
    for variant in ("low_prior", *LEARNED_VARIANTS):
        predict_controlled_variant(
            repository=ROOT,
            config_path=workflow_path,
            config=workflow,
            dataset_directory=dataset_dir,
            experiment_directory=experiment_dir,
            variant=variant,
        )
else:
    print("Evaluation is pending unless complete matched checkpoints/predictions already exist.")
    print("Set SAGE_AVO_RUN_EVALUATION=1 after Notebook 04 production training.")

### A2. Metrics and non-cherry-picked representative selection

In [ ]:
prediction_manifests = [
    experiment_dir / "predictions" / variant / "manifest.json"
    for variant in ("low_prior", *LEARNED_VARIANTS)
]
metrics_available = all(path.exists() for path in prediction_manifests)
if metrics_available:
    summary, per_realization, paired, representative_id = evaluate_controlled_ablation(
        experiment_directory=experiment_dir,
        dataset_directory=dataset_dir,
        bootstrap_repetitions=int(workflow["evaluation"]["bootstrap_repetitions"]),
        bootstrap_confidence=float(workflow["evaluation"]["bootstrap_confidence"]),
        seed=int(workflow["experiment"]["seed"]),
    )
    summary.to_csv(experiment_dir / "controlled_summary.csv", index=False)
    per_realization.to_csv(experiment_dir / "controlled_per_realization.csv", index=False)
    paired.to_csv(experiment_dir / "controlled_paired_improvements.csv", index=False)
    display(summary)
    display(paired)
    print("Representative test realization (median full-model Vp RMSE):", representative_id)
else:
    summary = per_realization = paired = pd.DataFrame()
    representative_id = None
    display(pd.DataFrame(columns=["variant", "domain", "metric", "mean", "std", "n_realizations"]))
    print("No controlled numerical claims are emitted until every matched prediction manifest exists.")

## Part B — Scientifically interpretable graph evidence

The representative synthetic realization is fixed by median full-model Vp RMSE—not visual appeal. Panel (f) displays only the strongest 15% of graph edges ranked by the **actual edge weight used during message passing**. This threshold is visualization only; all graph edges remain active in the network.

The graph-benefit panel is

\[
|Vp_{noGNN}-Vp_{true}|-|Vp_{full}-Vp_{true}|,
\]

so positive values mean graph propagation reduces absolute error. Latent embedding norm is not used as the primary scientific evidence.

In [ ]:
if metrics_available:
    realization_path = dataset_dir / "realizations" / f"realization_{representative_id:07d}.npz"
    with np.load(realization_path) as archive:
        avo = archive["avo"]
        rgt = archive["rgt"]
        truth = archive["elastic"]
    with np.load(experiment_dir / "predictions" / "full" / f"realization_{representative_id:07d}.npz") as archive:
        full_prediction = archive["elastic"]
    with np.load(experiment_dir / "predictions" / "no_gnn" / f"realization_{representative_id:07d}.npz") as archive:
        no_gnn_prediction = archive["elastic"]

    normalized_avo = torch.from_numpy(avo[None].astype(np.float32))
    _, gradient = angular_features(normalized_avo)
    edge_index = build_rgt_edges(torch.from_numpy(rgt[None].astype(np.float32)), max_shift=3, steered=True)[0]
    flat_gradient = gradient[0, 0].flatten()
    contrast = torch.abs(flat_gradient[edge_index[0]] - flat_gradient[edge_index[1]])
    weight = torch.exp(-contrast / (contrast.std() + 1e-6))
    graph_edges = GraphEdges(
        source=edge_index[0].numpy(), destination=edge_index[1].numpy(), weight=weight.numpy()
    )
    figure = plot_graph_mechanism(
        avo, rgt, gradient[0, 0].numpy(), graph_edges,
        full_prediction[0], no_gnn_prediction[0], truth[0],
    )
    graph_path = figure_dir / "stage05_graph_mechanism.png"
    figure.savefig(graph_path, dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Graph-benefit figure pending matched full/no-GNN predictions; no surrogate embedding plot is substituted.")

## Part C — Whole-image synthetic inference

In [ ]:
if metrics_available:
    with np.load(realization_path) as archive:
        low_prior = archive["low"]
    figure = plot_inversion_comparison(truth, low_prior, full_prediction)
    whole_path = figure_dir / "stage05_whole_image_synthetic.png"
    figure.savefig(whole_path, dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Truth/prior/prediction/error figure pending the controlled full-model checkpoint.")

## Part D — Field deployment and QC

The deployment input comprises real low/mid/high AVO stacks, Stage-01 RGT, and the Stage-01 field-conditioned low-frequency Vp/Vs/density model. The SEG-Y export axis remains described as the **configured gather-coordinate header** until acquisition/export metadata independently establish that it is true incidence angle.

Whole-section inference uses the same normalization, patch size, overlap, Hann stitching, and deterministic transport integrator as synthetic evaluation. Local well curves are overlays for field consistency assessment; because they contributed upstream, they are not blind validation.

In [ ]:
dataset_id = workflow["field_application"]["dataset_id"]
version = workflow["field_application"]["stage01_version"]
field_root = Path(paths["work_data_root"]) / dataset_id
field_files = {
    "avo_near": field_root / "usable" / version / "real_avo" / "AVO_low_real.npy",
    "avo_mid": field_root / "usable" / version / "real_avo" / "AVO_mid_real.npy",
    "avo_far": field_root / "usable" / version / "real_avo" / "AVO_high_real.npy",
    "low": field_root / "usable" / version / "elastic_background.npy",
    "rgt": field_root / "attributes" / version / "rgt_tau.npy",
    "time_ms": field_root / "usable" / version / "reg_t.npy",
    "cdp": field_root / "usable" / version / "good_cdps.npy",
    "line_xy": field_root / "usable" / version / "line_xy.npy",
}
missing_field = [path for path in field_files.values() if not path.exists()]
if missing_field:
    raise FileNotFoundError("Stage-01 field deployment channels are missing:\n" + "\n".join(map(str, missing_field)))
loaded_field = {name: np.load(path, allow_pickle=False) for name, path in field_files.items()}
field = {
    "avo": np.stack([loaded_field.pop("avo_near"), loaded_field.pop("avo_mid"), loaded_field.pop("avo_far")]),
    **loaded_field,
}
display(pd.DataFrame([{"channel": name, "shape": value.shape} for name, value in field.items()]))

fig, axes = plt.subplots(2, 4, figsize=(15, 7), constrained_layout=True)
for axis, panel, title in zip(
    axes.flat,
    [*field["avo"], field["rgt"], *field["low"], field["low"][0] - field["low"][0].mean(axis=0)],
    ["Real near AVO", "Real mid AVO", "Real far AVO", "RGT", "Low Vp", "Low Vs", "Low density", "Vp vertical variation"],
):
    axis.imshow(panel, aspect="auto", cmap="gray" if "AVO" in title else "viridis")
    axis.set_title(title); axis.set_xticks([]); axis.set_yticks([])
field_input_path = figure_dir / "stage05_field_input_contract.png"
fig.savefig(field_input_path, dpi=300, bbox_inches="tight")
plt.show()

### D1. Whole-section checkpoint inference

In [ ]:
field_prediction = field_segmentation = None
if checkpoints["full"].exists():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_controlled_model("full", workflow, checkpoints["full"], device)
    field_prediction, field_segmentation = infer_full_realization(
        model,
        avo=field["avo"], low=field["low"], rgt=field["rgt"],
        normalization=load_normalization(dataset_dir),
        patch_shape=tuple(workflow["patches"]["shape"]),
        stride=tuple(workflow["patches"]["stride"]),
        steps=int(workflow["training"]["sample_steps_test"]),
        batch_size=int(workflow["training"]["batch_size"]),
        device=device,
    )
    np.savez_compressed(
        private_root / "stage_artifacts" / "stage05_field_prediction.npz",
        elastic=field_prediction, segmentation=field_segmentation,
        time_ms=field["time_ms"], cdp=field["cdp"],
    )
else:
    print("Field prediction pending the controlled full-model checkpoint; the field inputs are not replaced by toy data.")

### D2. Wells, forward seismic QC, and far-angle behavior

When a prediction exists, local processed wells are overlaid in time/CDP coordinates. Exact forward-modeled near/mid/far stacks from the predicted elastic section are compared with real stacks using per-band robust amplitude fitting and correlation. Near, mid, and far statistics are reported separately; weak far-angle agreement is retained and discussed rather than hidden.

In [ ]:
if field_prediction is not None:
    well_qc, well_overlays = field_well_consistency(
        field_prediction,
        time_ms=field["time_ms"],
        line_xy=field["line_xy"],
        wells_directory=field_root / "usable" / version / "wells",
    )
    display(well_qc)
    well_qc.to_csv(private_root / "stage_artifacts" / "stage05_field_well_consistency.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    for channel, (axis, name) in enumerate(zip(axes, ("Vp", "Vs", "density"))):
        image = axis.imshow(
            field_prediction[channel], aspect="auto", cmap="viridis",
            extent=[0, field_prediction.shape[2] - 1, field["time_ms"][-1], field["time_ms"][0]],
        )
        for overlay in well_overlays:
            if overlay["channel"] == channel:
                axis.scatter(
                    np.full_like(overlay["time_ms"], overlay["trace_index"]),
                    overlay["time_ms"], c=overlay["observed"], cmap="viridis",
                    vmin=np.nanpercentile(field_prediction[channel], 2),
                    vmax=np.nanpercentile(field_prediction[channel], 98), s=2,
                )
        axis.set(title=f"Predicted {name} with processed-well overlays", xlabel="Trace", ylabel="TWT (ms)")
        fig.colorbar(image, ax=axis, shrink=0.75)
    well_path = figure_dir / "stage05_field_prediction_with_wells.png"
    fig.savefig(well_path, dpi=300, bbox_inches="tight")
    plt.show()

    modeled = forward_avo_three_band(*field_prediction, config=ForwardConfig())
    agreement = compare_forward_outputs(field["avo"], modeled)
    forward_qc = pd.DataFrame({
        "band": ("near", "mid", "far"),
        "fitted_amplitude_scale": agreement.scale,
        "correlation": agreement.correlation,
        "normalized_rmse_after_scale": agreement.normalized_rmse,
    })
    display(forward_qc)
    forward_qc.to_csv(private_root / "stage_artifacts" / "stage05_field_forward_qc.csv", index=False)
else:
    print("Forward field QC is pending; no correlation values are fabricated.")

### D3. Model/prior sensitivity

Checkpoint ensembles, alternative justified prior cutoffs, and controlled model variants may be propagated through the same inference code. Their spread is reported as **model/prior sensitivity**, **predictive sensitivity**, or **ensemble sensitivity**. It is not a calibrated posterior uncertainty distribution.

In [ ]:
sensitivity_dir = experiment_dir / "field_sensitivity"
run_sensitivity = os.getenv("SAGE_AVO_RUN_FIELD_SENSITIVITY", "0") == "1"
if run_sensitivity:
    sensitivity_dir.mkdir(parents=True, exist_ok=True)
    if not all_checkpoints_available:
        raise FileNotFoundError("Matched controlled checkpoints are required for model-variant sensitivity.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for variant, checkpoint in checkpoints.items():
        model = load_controlled_model(variant, workflow, checkpoint, device)
        member, _ = infer_full_realization(
            model,
            avo=field["avo"], low=field["low"], rgt=field["rgt"],
            normalization=load_normalization(dataset_dir),
            patch_shape=tuple(workflow["patches"]["shape"]),
            stride=tuple(workflow["patches"]["stride"]),
            steps=int(workflow["training"]["sample_steps_test"]),
            batch_size=int(workflow["training"]["batch_size"]),
            device=device,
        )
        np.savez_compressed(sensitivity_dir / f"member_{variant}.npz", elastic=member)

sensitivity_files = sorted(sensitivity_dir.glob("member_*.npz"))
if len(sensitivity_files) >= 2:
    members = np.stack([np.load(path)["elastic"] for path in sensitivity_files])
    sensitivity = ensemble_sensitivity(members)
    print("Sensitivity members:", len(members), "shape:", sensitivity["standard_deviation"].shape)
    np.savez_compressed(sensitivity_dir / "model_variant_sensitivity_summary.npz", **sensitivity)
else:
    print("At least two predeclared checkpoint/prior/model variants are required; sensitivity maps remain pending.")
    print("Set SAGE_AVO_RUN_FIELD_SENSITIVITY=1 after controlled training.")

## Stage outputs

| artifact | shape/type | scientific meaning | consumed by |
|---|---|---|---|
| controlled prediction packages | full test images per variant | Matched whole-realization benchmark predictions | metric/figure pipeline |
| per-realization/summary/paired CSVs | metric tables | Performance and paired ablation evidence | paper/meeting tables |
| graph mechanism figure | 2×4 high-resolution panel | AVO/RGT/edge mechanism and graph error reduction | IMAGE/paper |
| whole-image synthetic figure | truth/prior/prediction/error for Vp/Vs/density | Spatial inversion performance | IMAGE/paper |
| field prediction/QC package | whole section + coordinates + QC | Field deployment and consistency assessment | IMAGE/paper |
| sensitivity maps | ensemble spread | Model/prior sensitivity, not posterior uncertainty | discussion |

## Scientific checks

- Numerical tables require all matched controlled prediction manifests; absent results remain empty, not relabeled historical values.
- Metrics are calculated per realization and paired by realization before aggregation.
- The representative case is chosen by median full-model Vp RMSE.
- Only the strongest 15% of actual edge weights are drawn; all edges remain active in message passing.
- Whole-image inference uses common tiling, overlap, integration, normalization, and checkpoint rules.
- Field wells are treated as non-blind consistency overlays.
- Forward QC preserves band-specific behavior, including weak far-angle agreement.
- Ensemble spread is labeled sensitivity rather than calibrated posterior uncertainty.

## Next stage

This is the final notebook. Its artifacts feed the IMAGE presentation and manuscript only after redistribution rights and controlled-run completeness are verified in the private figure index.